In [ ]:
# Cell 1: Configuration
import sys, os, shutil

# === CONFIGURE THESE ===
PARQUET_DIR = "/kaggle/input/datasets/vitorandtxr/cryptobot-binance"
EXPERIMENT  = "multi_pair_v2_kaggle"  # config name under configs/experiments/
RESUME_FROM = None  # set to checkpoint path to resume, e.g. f"{PARQUET_DIR}/best_model.pt"
POS_WEIGHT  = None  # set to override pos_weight, e.g. 5.0 (None = use config value)
# =======================

DATA_DATASET = PARQUET_DIR
WORK_DIR = "/kaggle/working/SignalCortex"

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
shutil.copytree(f"{DATA_DATASET}/SignalCortex", WORK_DIR)

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

# Redirect outputs to persistent Kaggle location
import yaml
config_path = f"configs/experiments/{EXPERIMENT}.yaml"
with open(config_path) as f:
    cfg = yaml.safe_load(f)
cfg["export"]["output_dir"] = "/kaggle/working/outputs/"
with open(config_path, "w") as f:
    yaml.dump(cfg, f)

print(f"Working dir: {os.getcwd()}")
print(f"Experiment:  {EXPERIMENT}")
print(f"Parquet dir: {PARQUET_DIR}")
print(f"Resume from: {RESUME_FROM}")
print(f"Pos weight:  {POS_WEIGHT or 'from config'}")
print(f"GPU: {os.popen('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader').read().strip()}")

In [ ]:
# Cell 2: Install dependencies
!pip install -q tensorboard tqdm

In [ ]:
# Cell 3: Train
resume_flag = f"--resume {RESUME_FROM}" if RESUME_FROM else ""
pw_flag = f"--pw {POS_WEIGHT}" if POS_WEIGHT is not None else ""
!python main.py train \
    --config configs/experiments/{EXPERIMENT}.yaml \
    --parquet {PARQUET_DIR} \
    {resume_flag} {pw_flag}

In [ ]:
# Cell 4: Evaluate best model
!python main.py evaluate \
    --config configs/experiments/{EXPERIMENT}.yaml \
    --checkpoint /kaggle/working/outputs/best_model.pt \
    --parquet {PARQUET_DIR}

In [ ]:
# Cell 5: Backtest per pair
BACKTEST_PAIRS = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT"]
for pair in BACKTEST_PAIRS:
    print(f"\n{'='*60}")
    print(f"Backtesting {pair}")
    print(f"{'='*60}")
    !python main.py backtest \
        --config configs/experiments/{EXPERIMENT}.yaml \
        --checkpoint /kaggle/working/outputs/best_model.pt \
        --pair {pair} \
        --parquet {PARQUET_DIR}

In [ ]:
# Cell 6: List saved outputs
for root, dirs, files in os.walk("/kaggle/working/outputs"):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path) / 1024 / 1024
        print(f"{path} ({size:.1f} MB)")